# Fase 4 - Feature Engineering: Construyendo las Variables para ML

**Pregunta de negocio:** ¿Qué features representan mejor un viaje en taxi?

## Objetivos de este notebook

1. Cargar una muestra limpia de ~200K viajes desde BigQuery preparada para ML
2. Crear features temporales: hora, día, fin de semana, hora punta, noche, mes
3. Incorporar un feature de feriados (principales feriados de EE.UU. en 2015)
4. Features geográficos: clasificar zonas y aeropuertos
5. Features del viaje: velocidad, tarifa por milla, tarifa por minuto
6. Aplicar encoding categórico: OneHot, Ordinal
7. Comparar escalado: StandardScaler vs MinMaxScaler
8. Selección de features: correlación, varianza mínima, información mutua
9. Construir la matriz final X y los targets (fare_amount, tip_amount, tip_pct)
10. Guardar dataset procesado en formato parquet

## ¿Por qué es crítico el Feature Engineering?

Los modelos de ML solo son tan buenos como los datos que reciben. Transformar datos
crudos en features informativos es el paso que más impacto tiene en el rendimiento
del modelo. Un buen feature engineering puede superar fácilmente la diferencia entre
un modelo simple y uno complejo.

## 1. Configuración e importaciones

In [ ]:
# Conexión a BigQuery
import sys
sys.path.insert(0, '../../../src')
from bigquery.bq_helper import BigQueryHelper
bq = BigQueryHelper()

In [ ]:
# Librerías de análisis y ML
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Scikit-learn: preprocesamiento y selección de features
from sklearn.preprocessing import (
    OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler
)
from sklearn.feature_selection import (
    VarianceThreshold, mutual_info_regression
)

import warnings
warnings.filterwarnings('ignore')

# Configuración de estilo
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

## 2. Carga de datos desde BigQuery (~200K viajes)

Extraemos una muestra más grande que en fases anteriores porque los modelos de ML
se benefician de más datos. Incluimos filtros de calidad estrictos para evitar
registros problemáticos que distorsionen el entrenamiento.

In [ ]:
query = """
SELECT
    pickup_datetime,
    dropoff_datetime,
    passenger_count,
    trip_distance,
    pickup_location_id,
    dropoff_location_id,
    payment_type,
    fare_amount,
    tip_amount,
    tolls_amount,
    total_amount,
    rate_code,
    -- Columnas derivadas en SQL
    TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, SECOND) / 60.0 AS duration_min,
    CASE
        WHEN TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, SECOND) > 0
             AND trip_distance > 0
        THEN trip_distance / (TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, SECOND) / 3600.0)
        ELSE NULL
    END AS speed_mph,
    CASE
        WHEN fare_amount > 0
        THEN (tip_amount / fare_amount) * 100
        ELSE NULL
    END AS tip_pct
FROM
    `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`
WHERE
    fare_amount BETWEEN 2.5 AND 200
    AND trip_distance BETWEEN 0.1 AND 50
    AND passenger_count BETWEEN 1 AND 6
    AND pickup_location_id IS NOT NULL
    AND dropoff_location_id IS NOT NULL
    AND TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, SECOND) BETWEEN 60 AND 7200
    AND tip_amount >= 0
    AND tolls_amount >= 0
ORDER BY RAND()
LIMIT 200000
"""

df = bq.query_to_df(query)
print(f"Registros cargados: {len(df):,}")
print(f"Columnas: {df.shape[1]}")
df.head()

In [ ]:
# Verificación rápida
print("Tipos de datos:")
print(df.dtypes)
print(f"\nValores nulos por columna:")
print(df.isnull().sum())
print(f"\nFilas con algún nulo: {df.isnull().any(axis=1).sum():,}")

# Eliminar filas con nulos en speed_mph o tip_pct
df = df.dropna(subset=['speed_mph', 'tip_pct']).copy()
print(f"\nRegistros tras eliminar nulos: {len(df):,}")

## 3. Features temporales

El comportamiento de los taxis varía drásticamente según el momento del día y la
semana. Extraemos componentes temporales que capturen estos patrones:

- **pickup_hour**: hora del día (0-23)
- **pickup_day_of_week**: día de la semana (0=lunes, 6=domingo)
- **is_weekend**: 1 si es sábado o domingo
- **is_rush_hour**: 1 si la hora está en rangos punta (7-9, 17-19)
- **is_night**: 1 si es horario nocturno (20-6)
- **month**: mes del año (1-12)

In [ ]:
# Asegurar tipo datetime
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
df['dropoff_datetime'] = pd.to_datetime(df['dropoff_datetime'])

# Features temporales
df['pickup_hour'] = df['pickup_datetime'].dt.hour
df['pickup_day_of_week'] = df['pickup_datetime'].dt.dayofweek  # 0=lunes
df['is_weekend'] = (df['pickup_day_of_week'] >= 5).astype(int)
df['is_rush_hour'] = df['pickup_hour'].apply(
    lambda h: 1 if (7 <= h <= 9) or (17 <= h <= 19) else 0
)
df['is_night'] = df['pickup_hour'].apply(
    lambda h: 1 if h >= 20 or h <= 6 else 0
)
df['month'] = df['pickup_datetime'].dt.month

print("Features temporales creados:")
print(df[['pickup_datetime', 'pickup_hour', 'pickup_day_of_week', 
          'is_weekend', 'is_rush_hour', 'is_night', 'month']].head(10))

In [ ]:
# Visualizar distribución de features temporales
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Distribución por hora
df['pickup_hour'].value_counts().sort_index().plot(kind='bar', ax=axes[0, 0], color='#2196F3')
axes[0, 0].set_title('Viajes por Hora de Recogida', fontweight='bold')
axes[0, 0].set_xlabel('Hora')
axes[0, 0].set_ylabel('Cantidad')

# Distribución por día de la semana
dias = ['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom']
day_counts = df['pickup_day_of_week'].value_counts().sort_index()
axes[0, 1].bar(range(7), day_counts.values, color='#4CAF50')
axes[0, 1].set_xticks(range(7))
axes[0, 1].set_xticklabels(dias)
axes[0, 1].set_title('Viajes por Día de la Semana', fontweight='bold')
axes[0, 1].set_ylabel('Cantidad')

# Weekend vs weekday
weekend_labels = ['Laborable', 'Fin de semana']
weekend_counts = df['is_weekend'].value_counts().sort_index()
axes[0, 2].bar(weekend_labels, weekend_counts.values, color=['#FF9800', '#9C27B0'])
axes[0, 2].set_title('Laborable vs Fin de Semana', fontweight='bold')
axes[0, 2].set_ylabel('Cantidad')

# Rush hour
rush_labels = ['Normal', 'Hora Punta']
rush_counts = df['is_rush_hour'].value_counts().sort_index()
axes[1, 0].bar(rush_labels, rush_counts.values, color=['#607D8B', '#F44336'])
axes[1, 0].set_title('Hora Normal vs Hora Punta', fontweight='bold')
axes[1, 0].set_ylabel('Cantidad')

# Nocturno
night_labels = ['Día', 'Noche']
night_counts = df['is_night'].value_counts().sort_index()
axes[1, 1].bar(night_labels, night_counts.values, color=['#FFC107', '#3F51B5'])
axes[1, 1].set_title('Viajes Diurnos vs Nocturnos', fontweight='bold')
axes[1, 1].set_ylabel('Cantidad')

# Por mes
df['month'].value_counts().sort_index().plot(kind='bar', ax=axes[1, 2], color='#E91E63')
axes[1, 2].set_title('Viajes por Mes', fontweight='bold')
axes[1, 2].set_xlabel('Mes')
axes[1, 2].set_ylabel('Cantidad')

plt.suptitle('Distribución de Features Temporales', fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

## 4. Feature de feriados

Los feriados afectan significativamente el comportamiento del tráfico y las propinas.
Definimos manualmente los principales feriados de EE.UU. en 2015.

In [ ]:
# Principales feriados de EE.UU. en 2015
us_holidays_2015 = [
    '2015-01-01',  # Año Nuevo
    '2015-01-19',  # Martin Luther King Jr. Day
    '2015-02-16',  # Presidents' Day
    '2015-05-25',  # Memorial Day
    '2015-07-03',  # Día de la Independencia (observado)
    '2015-07-04',  # Día de la Independencia
    '2015-09-07',  # Labor Day
    '2015-10-12',  # Columbus Day
    '2015-11-11',  # Veterans Day
    '2015-11-26',  # Thanksgiving
    '2015-11-27',  # Black Friday
    '2015-12-25',  # Navidad
    '2015-12-31',  # Nochevieja
]
us_holidays_2015 = pd.to_datetime(us_holidays_2015)

df['pickup_date'] = df['pickup_datetime'].dt.date
df['is_holiday'] = df['pickup_datetime'].dt.normalize().isin(us_holidays_2015).astype(int)

n_holiday = df['is_holiday'].sum()
print(f"Viajes en feriados: {n_holiday:,} ({n_holiday/len(df)*100:.1f}%)")

# Comparar tarifa promedio en feriados vs días normales
holiday_comparison = df.groupby('is_holiday').agg(
    fare_mean=('fare_amount', 'mean'),
    tip_mean=('tip_amount', 'mean'),
    distance_mean=('trip_distance', 'mean'),
    count=('fare_amount', 'count')
).round(2)
holiday_comparison.index = ['Día Normal', 'Feriado']
print("\nComparación feriados vs días normales:")
holiday_comparison

## 5. Features geográficos: zona y aeropuerto

Clasificamos cada viaje según la zona de recogida usando `pickup_location_id`
(TLC Zone ID, valores 1-263). Las ubicaciones de aeropuertos tienen tarifas
especiales (tarifa plana al JFK, por ejemplo).

In [ ]:
# Clasificar zonas usando pickup_location_id (TLC Zone IDs)
MANHATTAN_ZONES = {'4','12','13','24','41','42','43','45','48','50','68','74','75','79','87','88','90','100','107','113','114','116','125','127','128','137','140','141','142','143','144','148','151','152','153','158','161','162','163','164','166','170','186','194','202','209','211','224','229','230','231','232','233','234','236','237','238','239','243','244','246','249','261','262','263'}
BROOKLYN_ZONES = {'11','14','17','21','22','25','26','29','33','34','35','36','37','39','40','49','52','54','55','61','62','63','65','66','67','69','71','72','76','77','80','85','89','91','97','106','108','111','112','123','133','149','150','154','155','165','177','178','181','188','189','190','195','210','217','222','225','227','228','255','256','257'}
JFK_ZONE = {'132'}
LAGUARDIA_ZONE = {'138'}

def classify_zone(location_id):
    """
    Clasifica la ubicación en zonas basadas en pickup_location_id (TLC Zone ID).
    """
    loc = str(location_id)
    
    if loc in JFK_ZONE:
        return 'JFK'
    elif loc in LAGUARDIA_ZONE:
        return 'LaGuardia'
    elif loc in MANHATTAN_ZONES:
        return 'Manhattan'
    elif loc in BROOKLYN_ZONES:
        return 'Brooklyn'
    else:
        return 'Otro'


df['zone'] = df['pickup_location_id'].apply(classify_zone)
df['is_airport'] = df['zone'].isin(['JFK', 'LaGuardia']).astype(int)

print("Distribución por zona:")
zone_dist = df['zone'].value_counts()
for zone, count in zone_dist.items():
    print(f"  {zone:12s}: {count:>7,} ({count/len(df)*100:5.1f}%)")

print(f"\nViajes desde aeropuertos: {df['is_airport'].sum():,} ({df['is_airport'].mean()*100:.1f}%)")

In [ ]:
# Tarifa promedio por zona
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

zone_stats = df.groupby('zone').agg(
    fare_mean=('fare_amount', 'mean'),
    fare_median=('fare_amount', 'median'),
    tip_mean=('tip_amount', 'mean'),
    count=('fare_amount', 'count')
).sort_values('fare_mean', ascending=False)

zone_stats['fare_mean'].plot(kind='barh', ax=axes[0], color='#2196F3')
axes[0].set_title('Tarifa Promedio por Zona de Recogida', fontweight='bold')
axes[0].set_xlabel('Tarifa Promedio (USD)')

zone_stats['tip_mean'].plot(kind='barh', ax=axes[1], color='#FF9800')
axes[1].set_title('Propina Promedio por Zona de Recogida', fontweight='bold')
axes[1].set_xlabel('Propina Promedio (USD)')

plt.tight_layout()
plt.show()

print("\nEstadísticas por zona:")
zone_stats.round(2)

## 6. Features del viaje: velocidad, tarifa por milla/minuto

Creamos ratios que capturan la eficiencia del viaje. Estos features derivados
a menudo son más informativos que las variables originales por separado.

In [ ]:
# Features derivados del viaje
# speed_mph ya viene calculado desde SQL

# Tarifa por milla
df['fare_per_mile'] = np.where(
    df['trip_distance'] > 0,
    df['fare_amount'] / df['trip_distance'],
    np.nan
)

# Tarifa por minuto
df['fare_per_minute'] = np.where(
    df['duration_min'] > 0,
    df['fare_amount'] / df['duration_min'],
    np.nan
)

# Eliminar valores infinitos o extremos
for col in ['speed_mph', 'fare_per_mile', 'fare_per_minute']:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan)
    q99 = df[col].quantile(0.99)
    df[col] = df[col].clip(upper=q99)

# Rellenar nulos con la mediana
for col in ['speed_mph', 'fare_per_mile', 'fare_per_minute']:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

# Resumen
trip_features = ['speed_mph', 'fare_per_mile', 'fare_per_minute']
print("Estadísticas de features del viaje:")
df[trip_features].describe().round(2)

In [ ]:
# Visualizar distribución de features del viaje
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

trip_info = {
    'speed_mph': {'title': 'Velocidad Promedio', 'xlabel': 'mph', 'color': '#4CAF50'},
    'fare_per_mile': {'title': 'Tarifa por Milla', 'xlabel': 'USD/milla', 'color': '#2196F3'},
    'fare_per_minute': {'title': 'Tarifa por Minuto', 'xlabel': 'USD/min', 'color': '#FF9800'}
}

for ax, (col, info) in zip(axes, trip_info.items()):
    ax.hist(df[col], bins=50, color=info['color'], alpha=0.7, edgecolor='white')
    ax.axvline(df[col].mean(), color='red', linestyle='--', label=f"Media: {df[col].mean():.2f}")
    ax.axvline(df[col].median(), color='darkblue', linestyle='-', label=f"Mediana: {df[col].median():.2f}")
    ax.set_title(info['title'], fontweight='bold')
    ax.set_xlabel(info['xlabel'])
    ax.set_ylabel('Frecuencia')
    ax.legend()

plt.suptitle('Distribución de Features Derivados del Viaje', fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

## 7. Encoding de variables categóricas

Los modelos de ML necesitan entradas numéricas. Aplicamos distintas estrategias
según el tipo de variable:

- **OneHotEncoder**: para `zone` y `payment_type` (sin orden natural)
- **OrdinalEncoder**: para `rate_code` (tiene un orden implícito: estándar < JFK < Newark)

In [ ]:
# Preparar columnas categóricas
df['payment_type'] = df['payment_type'].astype(str)
df['rate_code'] = df['rate_code'].astype(str)

print("Valores únicos:")
print(f"  zone: {df['zone'].unique()}")
print(f"  payment_type: {df['payment_type'].unique()}")
print(f"  rate_code: {df['rate_code'].unique()}")

In [ ]:
# OneHotEncoder para zone
ohe_zone = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
zone_encoded = ohe_zone.fit_transform(df[['zone']])
zone_columns = [f'zone_{cat}' for cat in ohe_zone.categories_[0][1:]]  # drop='first'
df_zone = pd.DataFrame(zone_encoded, columns=zone_columns, index=df.index)

print("OneHot Encoding - zone:")
print(f"  Categorías originales: {ohe_zone.categories_[0]}")
print(f"  Columnas resultantes: {zone_columns}")
print(f"  Shape: {df_zone.shape}")
df_zone.head()

In [ ]:
# OneHotEncoder para payment_type
ohe_payment = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
payment_encoded = ohe_payment.fit_transform(df[['payment_type']])
payment_columns = [f'payment_{cat}' for cat in ohe_payment.categories_[0][1:]]
df_payment = pd.DataFrame(payment_encoded, columns=payment_columns, index=df.index)

print("OneHot Encoding - payment_type:")
print(f"  Categorías originales: {ohe_payment.categories_[0]}")
print(f"  Columnas resultantes: {payment_columns}")
df_payment.head()

In [ ]:
# OrdinalEncoder para rate_code
# 1=Estándar, 2=JFK, 3=Newark, 4=Nassau/Westchester, 5=Negociada, 6=Grupo
rate_order = [['1', '2', '3', '4', '5', '6']]
ord_encoder = OrdinalEncoder(categories=rate_order, handle_unknown='use_encoded_value',
                              unknown_value=-1)
df['rate_code_encoded'] = ord_encoder.fit_transform(df[['rate_code']])

print("Ordinal Encoding - rate_code:")
print(df[['rate_code', 'rate_code_encoded']].drop_duplicates().sort_values('rate_code_encoded'))

## 8. Feature Scaling: StandardScaler vs MinMaxScaler

Algunos modelos (regresión lineal regularizada, SVM, KNN) son sensibles a la escala
de las features. Comparamos dos enfoques:

- **StandardScaler**: centra en media 0 y escala a desviación estándar 1. Ideal para
  datos aproximadamente normales.
- **MinMaxScaler**: escala al rango [0, 1]. Preserva la forma de la distribución
  pero es sensible a outliers.

In [ ]:
# Features numéricos a escalar
numeric_features = [
    'trip_distance', 'duration_min', 'speed_mph', 'passenger_count',
    'fare_per_mile', 'fare_per_minute', 'pickup_hour', 'pickup_day_of_week', 'month'
]

# StandardScaler
standard_scaler = StandardScaler()
df_standard = pd.DataFrame(
    standard_scaler.fit_transform(df[numeric_features]),
    columns=[f'{col}_std' for col in numeric_features],
    index=df.index
)

# MinMaxScaler
minmax_scaler = MinMaxScaler()
df_minmax = pd.DataFrame(
    minmax_scaler.fit_transform(df[numeric_features]),
    columns=[f'{col}_mm' for col in numeric_features],
    index=df.index
)

print("Comparación de escalado (primeros 3 features):")
print("\nStandardScaler (media~0, std~1):")
print(df_standard.iloc[:, :3].describe().round(3))
print("\nMinMaxScaler (rango [0, 1]):")
print(df_minmax.iloc[:, :3].describe().round(3))

In [ ]:
# Visualizar el efecto del escalado
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
example_feat = 'trip_distance'

axes[0].hist(df[example_feat], bins=50, color='#2196F3', alpha=0.7, edgecolor='white')
axes[0].set_title(f'{example_feat} - Original', fontweight='bold')
axes[0].set_xlabel('Valor')

axes[1].hist(df_standard[f'{example_feat}_std'], bins=50, color='#4CAF50', alpha=0.7, edgecolor='white')
axes[1].set_title(f'{example_feat} - StandardScaler', fontweight='bold')
axes[1].set_xlabel('Valor escalado')

axes[2].hist(df_minmax[f'{example_feat}_mm'], bins=50, color='#FF9800', alpha=0.7, edgecolor='white')
axes[2].set_title(f'{example_feat} - MinMaxScaler', fontweight='bold')
axes[2].set_xlabel('Valor escalado')

plt.suptitle('Efecto del Escalado en trip_distance', fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print("Observación: Ambos escaladores preservan la forma de la distribución.")
print("StandardScaler es preferible para modelos lineales regularizados (Ridge, Lasso).")
print("MinMaxScaler es útil para redes neuronales y algoritmos basados en distancia.")

## 9. Selección de features

No todos los features son igualmente útiles. Aplicamos tres métodos de selección:

1. **Correlación con el target**: descartamos features con correlación muy baja
2. **Variance Threshold**: eliminamos features con varianza casi nula (constantes)
3. **Información Mutua**: mide la dependencia (lineal y no lineal) entre feature y target

In [ ]:
# Construir DataFrame con todos los features candidatos
feature_cols_numeric = [
    'trip_distance', 'duration_min', 'speed_mph', 'passenger_count',
    'fare_per_mile', 'fare_per_minute',
    'pickup_hour', 'pickup_day_of_week', 'month',
    'is_weekend', 'is_rush_hour', 'is_night', 'is_holiday', 'is_airport',
    'rate_code_encoded'
]

df_features = pd.concat([
    df[feature_cols_numeric].reset_index(drop=True),
    df_zone.reset_index(drop=True),
    df_payment.reset_index(drop=True)
], axis=1)

print(f"Matriz de features candidatos: {df_features.shape}")
print(f"Columnas: {list(df_features.columns)}")

In [ ]:
# 9.1 Correlación con el target (fare_amount)
target = df['fare_amount'].reset_index(drop=True)

correlations = df_features.corrwith(target).abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
correlations.plot(kind='barh', ax=ax, color='#2196F3')
ax.set_title('Correlación Absoluta de Features con fare_amount', fontweight='bold')
ax.set_xlabel('|Correlación de Pearson|')
ax.axvline(x=0.05, color='red', linestyle='--', label='Umbral mínimo (0.05)')
ax.legend()
plt.tight_layout()
plt.show()

low_corr = correlations[correlations < 0.05]
print(f"\nFeatures con correlación < 0.05 (candidatos a eliminar):")
for feat, corr in low_corr.items():
    print(f"  {feat}: {corr:.4f}")

In [ ]:
# 9.2 Variance Threshold
vt = VarianceThreshold(threshold=0.01)
vt.fit(df_features)

low_variance = pd.Series(vt.variances_, index=df_features.columns)
dropped_by_variance = low_variance[low_variance < 0.01]

print("Features con varianza < 0.01:")
if len(dropped_by_variance) == 0:
    print("  Ninguno - todos los features tienen varianza suficiente.")
else:
    for feat, var in dropped_by_variance.items():
        print(f"  {feat}: varianza = {var:.6f}")

print(f"\nFeatures que pasan el filtro de varianza: {vt.get_support().sum()} / {len(df_features.columns)}")

In [ ]:
# 9.3 Información Mutua con fare_amount
# Usamos una submuestra para velocidad (MI es costoso computacionalmente)
sample_idx = np.random.RandomState(42).choice(len(df_features), size=20000, replace=False)
X_sample = df_features.iloc[sample_idx]
y_sample = target.iloc[sample_idx]

mi_scores = mutual_info_regression(X_sample, y_sample, random_state=42)
mi_series = pd.Series(mi_scores, index=df_features.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
mi_series.plot(kind='barh', ax=ax, color='#4CAF50')
ax.set_title('Información Mutua de Features con fare_amount', fontweight='bold')
ax.set_xlabel('Información Mutua (bits)')
plt.tight_layout()
plt.show()

print("\nTop 10 features por Información Mutua:")
for i, (feat, score) in enumerate(mi_series.head(10).items(), 1):
    print(f"  {i:2d}. {feat:25s}: {score:.4f}")

In [ ]:
# Mapa de calor de correlaciones entre features (detectar multicolinealidad)
fig, ax = plt.subplots(figsize=(14, 12))
corr_matrix = df_features.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix.astype(float), mask=mask, annot=True, fmt='.2f', cmap='coolwarm',            center=0, ax=ax, square=True, linewidths=0.5,
            annot_kws={'size': 7})
ax.set_title('Matriz de Correlación entre Features', fontweight='bold')
plt.tight_layout()
plt.show()

# Detectar pares con alta correlación
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.7:
            high_corr_pairs.append((
                corr_matrix.columns[i],
                corr_matrix.columns[j],
                corr_matrix.iloc[i, j]
            ))

if high_corr_pairs:
    print("\nPares de features con correlación > 0.7 (posible multicolinealidad):")
    for f1, f2, corr in high_corr_pairs:
        print(f"  {f1} <-> {f2}: {corr:.3f}")
else:
    print("\nNo se detectaron pares con correlación > 0.7")

## 10. Matriz final X y targets

Construimos la matriz de features final seleccionando las variables más relevantes
basándonos en los análisis anteriores. Definimos tres posibles targets:

- **fare_amount**: tarifa del viaje (regresión, problema "fácil")
- **tip_amount**: propina (regresión, problema "difícil")
- **tip_pct**: porcentaje de propina sobre tarifa

In [ ]:
# Seleccionamos los features finales basados en el análisis
# Excluimos features con correlación muy baja y alta multicolinealidad

selected_features = [
    # Features del viaje
    'trip_distance', 'duration_min', 'speed_mph', 'passenger_count',
    'fare_per_mile', 'fare_per_minute',
    # Features temporales
    'pickup_hour', 'pickup_day_of_week', 'month',
    'is_weekend', 'is_rush_hour', 'is_night', 'is_holiday',
    # Features geográficos
    'is_airport', 'rate_code_encoded'
]

# Agregar columnas one-hot
selected_features_all = selected_features + zone_columns + payment_columns

X = pd.concat([
    df[selected_features].reset_index(drop=True),
    df_zone.reset_index(drop=True),
    df_payment.reset_index(drop=True)
], axis=1)

# Targets
y_fare = df['fare_amount'].reset_index(drop=True)
y_tip = df['tip_amount'].reset_index(drop=True)
y_tip_pct = df['tip_pct'].reset_index(drop=True)

print(f"Matriz de features X: {X.shape}")
print(f"Features seleccionados: {list(X.columns)}")
print(f"\nTargets:")
print(f"  fare_amount:  media={y_fare.mean():.2f}, mediana={y_fare.median():.2f}")
print(f"  tip_amount:   media={y_tip.mean():.2f}, mediana={y_tip.median():.2f}")
print(f"  tip_pct:      media={y_tip_pct.mean():.2f}, mediana={y_tip_pct.median():.2f}")

In [ ]:
# Verificación final: no debe haber nulos ni infinitos
print("Verificación de calidad de la matriz X:")
print(f"  Nulos: {X.isnull().sum().sum()}")
print(f"  Infinitos: {np.isinf(X.select_dtypes(include=[np.number])).sum().sum()}")
print(f"  Shape: {X.shape}")

# Si hay nulos, rellenar con la mediana
if X.isnull().sum().sum() > 0:
    print("\n  Rellenando nulos con la mediana...")
    X = X.fillna(X.median())
    print(f"  Nulos después de rellenar: {X.isnull().sum().sum()}")

X.describe().round(3)

## 11. Guardar dataset procesado

Guardamos la matriz de features y los targets en formato parquet para reutilizar
en los notebooks de regresión (02 y 03) sin necesidad de repetir el preprocesamiento.

In [ ]:
import os

# Crear directorio de salida si no existe
output_dir = '../../../data/processed/nyc_taxi'
os.makedirs(output_dir, exist_ok=True)

# Combinar features y targets en un solo DataFrame
df_final = X.copy()
df_final['fare_amount'] = y_fare.values
df_final['tip_amount'] = y_tip.values
df_final['tip_pct'] = y_tip_pct.values

# Guardar en parquet
output_path = os.path.join(output_dir, 'features_engineered.parquet')
df_final.to_parquet(output_path, index=False)

print(f"Dataset guardado en: {output_path}")
print(f"Shape: {df_final.shape}")
print(f"Tamaño en disco: {os.path.getsize(output_path) / 1024 / 1024:.1f} MB")

# Verificar lectura
df_check = pd.read_parquet(output_path)
print(f"\nVerificación de lectura: {df_check.shape}")
print(f"Columnas: {list(df_check.columns)}")

## Resumen del Feature Engineering

| Categoría | Features Creados | Descripción |
|-----------|-----------------|-------------|
| Temporales | pickup_hour, pickup_day_of_week, month, is_weekend, is_rush_hour, is_night | Capturan patrones horarios y semanales |
| Feriados | is_holiday | Feriados principales de EE.UU. 2015 |
| Geográficos | zone (one-hot), is_airport | Zonas de NYC y aeropuertos |
| Viaje | speed_mph, fare_per_mile, fare_per_minute | Ratios de eficiencia del viaje |
| Categóricos | payment_type (one-hot), rate_code (ordinal) | Encoding numérico |

### Hallazgos clave

1. **trip_distance y duration_min** son los features más correlacionados con fare_amount.
2. **fare_per_mile y fare_per_minute** tienen alta información mutua con el target.
3. Los features temporales (hora, día) aportan información moderada.
4. La zona de recogida (especialmente aeropuertos) captura tarifa planas especiales.

### Próximos pasos

- **Notebook 02**: Regresión para predecir fare_amount (problema "fácil")
- **Notebook 03**: Regresión para predecir tip_amount (problema "difícil")